# 01 — Circuit Patching

Load the latest `circuit_patching` run from the SQLite store, render the
recovery-fraction table, plot recovery vs layer, and discuss what causal
evidence means in this context.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Add src to path when running outside the installed package.
repo_root = Path("__file__").parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

In [ ]:
from mech_interp.storage.sqlite_store import SQLiteResultStore

store = SQLiteResultStore(
    database_path=repo_root / "artifacts" / "results.db",
    artifact_dir=repo_root / "artifacts",
)

runs = store.list_runs(limit=100)
cp_runs = [r for r in runs if r.family == "circuit_patching"]

if not cp_runs:
    raise RuntimeError(
        "No circuit_patching runs found. "
        "Run: mech run --name circuit_patching"
    )

latest = cp_runs[0]  # list_runs is DESC by id
result = store.get_result(latest.id)
print(f"Run {latest.id} | {latest.spec_name} | status={latest.status}")
print("Metrics:", json.dumps(result.metrics if result else {}, indent=2))

## Recovery-fraction table

Each row is a `(hook_site, layer)` patch.  `recovery` is the fraction of
the clean logit-diff recovered after patching from the corrupted run into
the clean run at that site.  Recovery ≈ 1 means the patch "fixes" the
model; recovery ≈ 0 means the site carries no causal information for the task.

In [ ]:
if result is None:
    raise RuntimeError(f"No result stored for run {latest.id}")

patch_json_path = result.artifacts.get("patch_results")
if patch_json_path is None:
    # Fall back: read all metrics that look like recovery scores.
    rows = [
        {"metric": k, "value": v}
        for k, v in result.metrics.items()
    ]
    df = pd.DataFrame(rows)
else:
    patch_data = json.loads(Path(patch_json_path).read_text())
    rows = patch_data if isinstance(patch_data, list) else patch_data.get("patches", [])
    df = pd.DataFrame(rows)

print(df.to_string(index=False))

## Recovery vs layer

We plot mean recovery fraction per layer for each hook type.  A sharp peak
at a specific layer is the hallmark of a "critical" layer for this task.

In [ ]:
metrics = result.metrics

# Build a plottable structure from metrics keys like "layer_N_resid_pre_recovery".
import re

layer_recovery: dict[str, list[tuple[int, float]]] = {}
pattern = re.compile(r"layer_(\d+)_(.+?)_recovery")
for key, val in metrics.items():
    m = pattern.match(key)
    if m:
        layer, hook = int(m.group(1)), m.group(2)
        layer_recovery.setdefault(hook, []).append((layer, float(val)))

if not layer_recovery:
    # If metrics don't follow that pattern, show a bar chart of all metrics.
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(list(metrics.keys()), list(metrics.values()))
    ax.set_xlabel("metric")
    ax.set_ylabel("value")
    ax.set_title("Circuit patching — all metrics")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    for hook, pts in sorted(layer_recovery.items()):
        pts.sort()
        layers, recoveries = zip(*pts)
        ax.plot(layers, recoveries, marker="o", label=hook)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Recovery fraction")
    ax.set_title("Circuit patching — recovery vs layer")
    ax.legend()
    plt.tight_layout()
    plt.show()

## What "causal evidence" means

Activation patching (a.k.a. causal tracing) asks: *if I replace a node's
activation with the value it would have had on a different input, how much
does the model's output change?*

High recovery at a site → the site is **causally necessary** for the task.
Low recovery → the site is either redundant, or the relevant information
passes through a different path.

The MLP control in the circuit-patching experiment patches MLP layers on
the same prompts.  Near-zero MLP recovery (typically seen in factual-recall
tasks) suggests attention heads, not MLPs, carry the critical information —
a finding that motivates the edge-level ACDC work in `03_acdc_lite.ipynb`.